In [ ]:
%pip install pandas
# Nebraska Broadband Gap Analysis
# FCC Broadband Data Collection, Nebraska fixed broadband (Copper, Cable, Fiber, Licensed Fixed Wireless)
# Satellite left out on purpose since it technically covers every location
# Setup: installing pandas for working with the data as tables



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# First look: find the four CSV files and preview 5 rows to see the columns
# Each row is one provider offering one technology at one address
import pandas as pd
import glob

files = glob.glob("*.csv")
print(files)

df = pd.read_csv(files[0], nrows=5)
df 


['bdc_31_FibertothePremises_fixed_broadband_D25_19oct2026.csv', 'bdc_31_LicensedFixedWireless_fixed_broadband_D25_19oct2026.csv', 'bdc_31_Copper_fixed_broadband_D25_19oct2026.csv', 'bdc_31_Cable_fixed_broadband_D25_19oct2026.csv']


,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,1566983,131043,Pinpoint Communications Inc,1014977071,50,1000,1000,1,X,NE,310619647002031,88261b8ce5fffff
1,1566983,131043,Pinpoint Communications Inc,1014977072,50,1000,1000,1,X,NE,310619647002045,88261b8dd9fffff
2,1566983,131043,Pinpoint Communications Inc,1014977093,50,1000,1000,1,X,NE,310619647002031,88261b8ce5fffff
3,1566983,131043,Pinpoint Communications Inc,1014977113,50,1000,1000,1,X,NE,310619647002042,88261b8c33fffff
4,1566983,131043,Pinpoint Communications Inc,1014977117,50,1000,1000,1,X,NE,310619647002026,88261b8c1bfffff


In [ ]:
# Load all four files into one table (3,458,420 rows)
# Reading block_geoid as text so I can pull the county code out of it later
# Technology counts confirm all four files loaded
frames = []
for f in files:
    frames.append(pd.read_csv(f, dtype={"block_geoid": str}))

df = pd.concat(frames, ignore_index = True)
print(df.shape)
df["technology"].value_counts()


(3458420, 12)


technology
71    1624678
50     866753
40     539848
10     427141
Name: count, dtype: int64

In [ ]:
# Clean and prepare
# Mapped technology codes to names: 10 Copper, 40 Cable, 50 Fiber, 71 Licensed Fixed Wireless
# Added a county column from the first 5 digits of the census block ID
# No missing values and no duplicate rows
# 781,151 unique locations; more rows than that because one address shows up once per provider/technology
tech_names = {10: "Copper", 40: "Cable", 50: "Fiber", 71: "Licensed Fixed Wireless"}
df["tech_name"] = df["technology"].map(tech_names)

df["county_fips"] = df["block_geoid"].str[:5]

print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Unique locations:", df["location_id"].nunique())

frn                              0
provider_id                      0
brand_name                       0
location_id                      0
technology                       0
max_advertised_download_speed    0
max_advertised_upload_speed      0
low_latency                      0
business_residential_code        0
state_usps                       0
block_geoid                      0
h3_res8_id                       0
tech_name                        0
county_fips                      0
dtype: int64
Duplicate rows: 0
Unique locations: 781151


In [ ]:
# Check speeds: shortened column names and looked at summary stats
# Median offering is 300 Mbps
# Problems: 295,916 rows (8.6%) report 0 Mbps, and a 200,000 Mbps max that's clearly a business connection
df = df.rename(columns={
    "max_advertised_download_speed": "download_mbps",
    "max_advertised_upload_speed": "upload_mbps",
    "download_mpbs": "download_mbps"
})

print(df[["download_mbps", "upload_mbps"]].describe())
print("Zero download speed rows:", (df["download_mbps"] == 0).sum())



       download_mbps   upload_mbps
count   3.458420e+06  3.458420e+06
mean    9.452556e+02  6.539569e+02
std     1.678942e+03  1.685518e+03
min     0.000000e+00  0.000000e+00
25%     2.500000e+01  3.000000e+00
50%     3.000000e+02  2.000000e+01
75%     1.000000e+03  1.000000e+02
max     2.000000e+05  2.000000e+05
Zero download speed rows: 295916


In [ ]:
# Where do the zeros come from?
# Almost all Licensed Fixed Wireless and Copper, mostly T-Mobile, CenturyLink, and AT&T
# 219 rows over 10,000 Mbps, leaving those out of speed averages as outliers
zeros = df[df["download_mbps"] == 0]

print(zeros["tech_name"].value_counts())
print(zeros["brand_name"].value_counts().head(10))
print("Rows over 10,000 Mbps:", (df["download_mbps"] > 10000).sum())


tech_name
Licensed Fixed Wireless    193572
Copper                     101265
Cable                         811
Fiber                         268
Name: count, dtype: int64
brand_name
T-Mobile                           161020
CenturyLink                         83557
AT&T                                32552
Great Plains Communications LLC     12393
Windstream Nebraska, Inc.            4698
Fort Randall Cable Systems, Inc       811
Pierce Telephone Co Inc               325
UPN                                   248
Consolidated                          148
K & M Telephone Company Inc           103
Name: count, dtype: int64
Rows over 10,000 Mbps: 219


In [ ]:
# Testing why they're zero
# First guess: speeds under 1 Mbps rounded down to 0
# But speeds are whole numbers, smallest nonzero speed is 10 Mbps, and every zero row also has 0 upload
# No slow DSL speeds like 3 or 6 Mbps exist, so it isn't rounding
# Looks like anything under about 10/1 Mbps is recorded as 0/0, so zeros are real but very slow service

print(df["download_mbps"].dtype)
print(sorted(df["download_mbps"].unique())[:10])
print(zeros["upload_mbps"].value_counts().head())


int64
[np.int64(0), np.int64(10), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(60), np.int64(70)]
upload_mbps
0    295916
Name: count, dtype: int64


In [ ]:
# Are people actually stuck with the zeros? Checked each location's best available option
# 259,024 locations have at least one 0 offering
# 243,150 of those (94%) also have a 100+ Mbps option
# Only 7,752 locations (~1% of the state) have nothing better than under 10 Mbps
# Decision: keep zeros for coverage questions, leave them out of speed averages
zero_locs = zeros["location_id"].unique()
best = df.groupby("location_id")["download_mbps"].max()
best_at_zero_locs = best.loc[zero_locs]

print("Locations with at least one 0 offering:", len(zero_locs))
print("Where 0 is the BEST option:", (best_at_zero_locs == 0).sum())
print("Where a 100+ Mbps option also exists:", (best_at_zero_locs >= 100).sum())

Locations with at least one 0 offering: 259024
Where 0 is the BEST option: 7752
Where a 100+ Mbps option also exists: 243150


In [ ]:
import sqlite3

cols = ["brand_name", "location_id", "tech_name", "download_mbps",
        "upload_mbps", "business_residential_code", "county_fips"]

conn = sqlite3.connect("ne_broadband.db")
df[cols].to_sql("availability", conn, if_exists="replace", index=False)

pd.read_sql("SELECT COUNT(*) AS row_count FROM availability", conn)


,row_count
0,3458420


In [ ]:
q1 = """
SELECT brand_name, tech_name, COUNT(DISTINCT location_id) AS locations
FROM availability
GROUP BY brand_name, tech_name
ORDER BY locations DESC
LIMIT 10
"""
pd.read_sql(q1, conn)


,brand_name,tech_name,locations
0,T-Mobile,Licensed Fixed Wireless,574989
1,Verizon,Licensed Fixed Wireless,498678
2,AT&T,Licensed Fixed Wireless,316636
3,Allo Communications LLC,Fiber,255446
4,Spectrum,Cable,253385
5,CenturyLink,Copper,244688
6,Cox Communications,Cable,216705
7,Nextlink,Licensed Fixed Wireless,125694
8,Quantum Fiber,Fiber,118824
9,"Windstream Nebraska, Inc.",Copper,91286


In [26]:
q2 = """
SELECT
    COUNT(DISTINCT location_id) AS total_locations,
    COUNT(DISTINCT CASE WHEN download_mbps >= 100 AND upload_mbps >= 20
                        THEN location_id END) AS served_100_20,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN download_mbps >= 100 AND upload_mbps >= 20
                        THEN location_id END) / COUNT(DISTINCT location_id), 1) AS pct_served,
    COUNT(DISTINCT CASE WHEN tech_name = 'Fiber' THEN location_id END) AS fiber_locations,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN tech_name = 'Fiber' THEN location_id END)
          / COUNT(DISTINCT location_id), 1) AS pct_fiber
FROM availability
"""
pd.read_sql(q2, conn)

,total_locations,served_100_20,pct_served,fiber_locations,pct_fiber
0,781151,740858,94.8,627799,80.4


In [ ]:
q3 = """
SELECT
    county_fips,
    COUNT(DISTINCT location_id) AS locations,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN download_mbps >= 100 AND upload_mbps >= 20
                        THEN location_id END) / COUNT(DISTINCT location_id), 1) AS pct_served,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN tech_name = 'Fiber' THEN location_id END)
          / COUNT(DISTINCT location_id), 1) AS pct_fiber
FROM availability
GROUP BY county_fips
ORDER BY pct_served ASC
LIMIT 10
"""
pd.read_sql(q3, conn)


,county_fips,locations,pct_served,pct_fiber
0,31087,1978,28.4,7.0
1,31149,1144,39.9,6.2
2,31085,616,40.7,39.0
3,31173,2607,44.4,5.4
4,31007,321,46.1,16.8
5,31117,243,66.3,39.5
6,31075,461,68.8,68.8
7,31165,542,69.6,22.0
8,31135,1652,70.5,25.5
9,31077,1489,72.7,27.7


In [ ]:
counties = pd.read_csv(
    "https://www2.census.gov/geo/docs/reference/codes2020/cou/st31_ne_cou2020.txt",
    sep="|", dtype=str
)
counties["county_fips"] = counties["STATEFP"] + counties["COUNTYFP"]
counties = counties[["county_fips", "COUNTYNAME"]].rename(columns={"COUNTYNAME": "county_name"})

counties.to_sql("counties", conn, if_exists="replace", index=False)
counties.head()


,county_fips,county_name
0,31001,Adams County
1,31003,Antelope County
2,31005,Arthur County
3,31007,Banner County
4,31009,Blaine County


In [ ]:
q4 = """
SELECT
    c.county_name,
    COUNT(DISTINCT a.location_id) AS locations,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.download_mbps >= 100 AND a.upload_mbps >= 20
                        THEN a.location_id END) / COUNT(DISTINCT a.location_id), 1) AS pct_served,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.tech_name = 'Fiber' THEN a.location_id END)
          / COUNT(DISTINCT a.location_id), 1) AS pct_fiber
FROM availability a
JOIN counties c ON a.county_fips = c.county_fips
GROUP BY c.county_name
ORDER BY pct_served ASC
LIMIT 10
"""
pd.read_sql(q4, conn)


,county_name,locations,pct_served,pct_fiber
0,Hitchcock County,1978,28.4,7.0
1,Rock County,1144,39.9,6.2
2,Hayes County,616,40.7,39.0
3,Thurston County,2607,44.4,5.4
4,Banner County,321,46.1,16.8
5,McPherson County,243,66.3,39.5
6,Grant County,461,68.8,68.8
7,Sioux County,542,69.6,22.0
8,Perkins County,1652,70.5,25.5
9,Greeley County,1489,72.7,27.7


In [ ]:
q5 = """
SELECT COUNT(*) AS copper_only_locations
FROM (
    SELECT location_id
    FROM availability
    GROUP BY location_id
    HAVING SUM(CASE WHEN brand_name = 'CenturyLink' AND tech_name = 'Copper' THEN 1 ELSE 0 END) > 0
       AND SUM(CASE WHEN tech_name IN ('Fiber', 'Cable') THEN 1 ELSE 0 END) = 0
) AS copper_only
"""
pd.read_sql(q5, conn)


,copper_only_locations
0,11943


In [31]:
q6 = """
SELECT c.county_name, COUNT(*) AS copper_only_locations
FROM (
    SELECT location_id, county_fips
    FROM availability
    GROUP BY location_id, county_fips
    HAVING SUM(CASE WHEN brand_name = 'CenturyLink' AND tech_name = 'Copper' THEN 1 ELSE 0 END) > 0
       AND SUM(CASE WHEN tech_name IN ('Fiber', 'Cable') THEN 1 ELSE 0 END) = 0
) AS copper_only
JOIN counties c ON copper_only.county_fips = c.county_fips
GROUP BY c.county_name
ORDER BY copper_only_locations DESC
LIMIT 10
"""
pd.read_sql(q6, conn)

,county_name,copper_only_locations
0,Douglas County,1486
1,Sarpy County,1378
2,Scotts Bluff County,1081
3,Dawes County,643
4,Madison County,546
5,Dodge County,532
6,Howard County,454
7,Lincoln County,445
8,Red Willow County,372
9,Morrill County,369
